# SARA small (44.9M) — Kaggle GPU Training on YOUR PDFs
**Kaggle setup:** Settings → Accelerator → **GPU T4 x2** ya P100. Internet: ON.

**Pehle ek baar (local):** apne PDFs wala folder zip karo:
```
zip -r drive_pdfs.zip sara_drive_data/
```
Phir Kaggle pe **+ Add Input → Upload → New Dataset** → zip upload → naam `drive-pdfs`

| Cell | Kaam | Time |
|---|---|---|
| 1 | Clone + install | 2 min |
| 2 | Extract your PDFs | 1 min |
| 3 | Corpus build (PDF→text→train.bin) | ~8 min |
| 4 | **Training 20k steps** | **~5-7 hr** T4 |
| 5 | Generate sample | 10s |
| 6 | Output save | 10s |

In [ ]:
# 1. Clone SARA repo + deps
!git clone https://github.com/skmandal3240/SARA /kaggle/working/SARA
%cd /kaggle/working/SARA
!pip install -q -r requirements.txt pyyaml pymupdf gdown

In [ ]:
# 2. Get the PDFs — OPTION A: attached Kaggle dataset
import glob, os, subprocess
zips = glob.glob('/kaggle/input/drive-pdfs/**/*.zip', recursive=True)
if zips:
    os.makedirs('/kaggle/working/pdfs', exist_ok=True)
    subprocess.run(['unzip','-o','-q',zips[0],'-d','/kaggle/working/pdfs'])
else:
    # OPTION B: public Drive folder link directly (folder must be 'Anyone with link')
    FOLDER_URL = 'https://drive.google.com/drive/folders/1OQk3FKLxI1GSVjf5ViW7nKu_x-oC3854'
    os.makedirs('/kaggle/working/pdfs', exist_ok=True)
    r = subprocess.run(['gdown','--folder',FOLDER_URL,'-O','/kaggle/working/pdfs'],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])

pdfs = glob.glob('/kaggle/working/pdfs/**/*.pdf', recursive=True)
print(f'{len(pdfs)} PDFs found')
assert pdfs, 'No PDFs! Check dataset attach / Drive permissions.'

In [ ]:
# 3. PDF -> text corpus -> train.bin (tokenizer retrains WITH your docs)
!python scripts/pdf_to_corpus.py /kaggle/working/pdfs
!python prepare_data.py --source file:data/drive_corpus.txt
# sanity: how much data do we have?
import os
print('corpus chars:', os.path.getsize('data/drive_corpus.txt'))
print('train.bin bytes:', os.path.getsize('data/train.bin'))

In [ ]:
# 4. TRAIN — 20k steps on T4 (~5-7 hr). Kaggle session limit 9h, fits.
#    Kam time: --steps 10000 (~2.5 hr). Resume nahi hota — ek hi run mein complete karo.
!python train_small.py --steps 20000 --batch 16

In [ ]:
# 5. Generate from YOUR trained model
import torch
from pathlib import Path
from generate import load_sara

model, tok, cfg = load_sara(Path('checkpoints/sara_small/sara.pt'))
for p in ['The sun is', 'Computer science is', 'Neural networks learn']:
    prompt = tok.wrap_user(p)
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long)
    out = model.generate(ids, max_new=60, temperature=0.8, eos_id=tok.eos_id)
    print(f'>>> {p}\n{tok.decode(out[0].tolist())}\n')

In [ ]:
# 6. Save output to /kaggle/working (auto-downloadable from Output tab)
import shutil
shutil.copy('checkpoints/sara_small/sara.pt', '/kaggle/working/sara_trained.pt')
shutil.copy('tokenizer/sara.json', '/kaggle/working/sara_tokenizer.json')
print('Done! Right panel -> Output tab -> download sara_trained.pt (~180 MB)')
print('Drive pe upload: https://drive.google.com -> New File Upload')